<a href="https://colab.research.google.com/github/rangedayo/first-repository/blob/main/rag_tutorial_reverse_hyde.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -q langchain==0.1.20 langchain-openai chromadb pypdf sentence_transformers tiktoken langchain-community

# 1. 에러를 유발하는 simsimd 삭제 및 안정적인 버전 재설치
!pip uninstall -y simsimd
!pip install -q "langchain==0.1.20" "langchain-openai" "chromadb" "pypdf" "tiktoken" "langchain-community==0.0.38" "numpy<2.0.0"

Found existing installation: simsimd 6.5.16
Uninstalling simsimd-6.5.16:
  Successfully uninstalled simsimd-6.5.16


In [ ]:
# 2. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 3. API 키 설정 (여기에 OpenAI API 키를 입력하세요)
import os
os.environ['OPENAI_API_KEY'] = 'sk'

In [ ]:
# 4. 모듈 임포트
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
import tiktoken

In [ ]:
# 5. 토크나이저 설정
tokenizer = tiktoken.get_encoding("cl100k_base")   # GPT-4, GPT-3.5-turbo 모델들이 사용하는 인코딩 방식

def tiktoken_len(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

In [ ]:
# 6. PDF 로드 및 분할
# 파일 경로가 맞는지 다시 한번 확인해주세요.
loader = PyPDFLoader("/content/drive/MyDrive/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=50,
    length_function=tiktoken_len
)
texts = text_splitter.split_documents(pages)

In [ ]:
# [핵심 로직] Reverse HyDE 구현 예시
from langchain.schema import Document

In [ ]:
# 7. 임베딩 및 벡터 스토어 생성
# HuggingFace 대신 OpenAI의 임베딩 모델을 사용합니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 1. 예상 질문을 뽑아낼 LLM 설정
question_gen_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 2. 각 텍스트 조각(texts)에 대해 예상 질문 생성
reverse_hyde_texts = []
for doc in texts:
    # LLM에게 예상 질문 3개를 뽑아달라고 함
    response = question_gen_llm.invoke(f"다음 내용을 읽고, 이 내용이 정답이 될 수 있는 질문 3개만 써줘:\n\n{doc.page_content}")

    # 원본 내용 + 예상 질문을 합쳐서 새로운 문서 객체 생성
    enhanced_content = f"질문들: {response.content}\n\n원본내용: {doc.page_content}"
    reverse_hyde_texts.append(Document(page_content=enhanced_content, metadata=doc.metadata))

    for i, doc in enumerate(texts):
    print(f"{i+1}/{len(texts)} 번째 조각 처리 중...")

# 3. 강화된 데이터를 벡터 DB에 저장
docsearch = Chroma.from_documents(reverse_hyde_texts, embedding_model)

docsearch = Chroma.from_documents(texts, embedding_model)

In [ ]:
# 8. OpenAI LLM 설정 (GPT-4o 또는 GPT-3.5-turbo 등)
llm_openai = ChatOpenAI(model="gpt-4o", temperature=0.0)

In [ ]:
# 9. RetrievalQA 체인 생성
qa = RetrievalQA.from_chain_type(
    llm=llm_openai,
    chain_type="stuff",
    retriever=docsearch.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10}
    )
)

In [ ]:
# 10. 질문 실행 및 결과 출력
query = "싱클레어는 누구니?"
result = qa.invoke(query)

from IPython.display import Markdown, display
display(Markdown(result["result"]))

싱클레어는 헤르만 헤세의 소설 "데미안"의 주인공입니다. 그는 소설에서 자신의 정체성을 찾고 성장하는 과정을 겪는 인물로, 데미안이라는 친구와의 관계를 통해 많은 영향을 받습니다.